# TinySSL Demo

Train a 3M-param ViT on a laptop CPU that matches DINOv2 on domain-specific tasks.

This notebook loads a pre-trained TinySSL model, runs inference on sample images, and visualizes attention maps.

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib
!git clone -q https://github.com/YOUR_USERNAME/TinySSL.git /content/TinySSL 2>/dev/null || true
import sys; sys.path.insert(0, '/content/TinySSL')

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
import requests
from io import BytesIO

from tinyssl.models.students import TinySSLBase

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')

## Load Pre-trained Model

In [ ]:
model = TinySSLBase()

# Option A: Load from local checkpoint
# ckpt = torch.load('checkpoints/flowers102/checkpoint_300.pt', map_location='cpu', weights_only=False)
# model.load_state_dict(ckpt['model'])

# Option B: Download from HuggingFace (uncomment when available)
# from huggingface_hub import hf_hub_download
# path = hf_hub_download(repo_id='YOUR_USERNAME/tinyssl-base', filename='checkpoint_300.pt')
# model.load_state_dict(torch.load(path, map_location='cpu', weights_only=False)['model'])

model.to(device).eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

## Load Sample Images

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Sample images from the web
urls = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/41/Sunflower_from_Silesia2.jpg/800px-Sunflower_from_Silesia2.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Image_created_with_a_mobile_phone.png/1200px-Image_created_with_a_mobile_phone.png',
]

images = []
originals = []
for url in urls:
    resp = requests.get(url, timeout=10)
    img = Image.open(BytesIO(resp.content)).convert('RGB')
    originals.append(img)
    images.append(transform(img))

batch = torch.stack(images).to(device)
print(f'Loaded {len(images)} images')

## Run Inference

In [ ]:
with torch.no_grad():
    output = model(batch)

cls_features = output['cls']  # [B, 256]
patch_features = output['patches']  # [B, N, 256]

print(f'CLS features shape: {cls_features.shape}')
print(f'Patch features shape: {patch_features.shape}')

# Cosine similarity between images
cls_norm = F.normalize(cls_features, dim=1)
sim_matrix = cls_norm @ cls_norm.T
print('\nImage similarity matrix:')
print(sim_matrix.cpu().numpy().round(3))

## Visualize Attention Maps

In [ ]:
def unnormalize(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(t.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(t.device)
    return (t * std + mean).clamp(0, 1)

def show_attention(model, image_tensor, original_img, idx=0):
    """Compute CLS-to-patch attention from the last encoder layer."""
    # Forward through encoder blocks manually to capture attention
    x = model.patch_embed(image_tensor.unsqueeze(0))
    x = x.flatten(2).transpose(1, 2)
    x = model.proj(x)
    cls = model.cls_token.expand(1, -1, -1)
    x = torch.cat([cls, x], dim=1)

    # Run encoder, get attention from last layer
    attn_weights = None
    for layer in model.encoder.layers:
        x = layer(x)

    # Use patch similarity to CLS as attention proxy
    cls_out = x[:, 0]
    patches = x[:, 1:]
    cls_norm = F.normalize(cls_out, dim=1)
    patch_norm = F.normalize(patches, dim=2)
    attn = (patch_norm * cls_norm.unsqueeze(1)).sum(2).squeeze(0)
    n = int(attn.shape[0] ** 0.5)
    attn_map = attn.reshape(n, n).detach().cpu()
    attn_map = F.interpolate(attn_map.unsqueeze(0).unsqueeze(0), size=(224, 224), mode='bilinear').squeeze()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(original_img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(original_img)
    axes[1].imshow(attn_map.numpy(), cmap='jet', alpha=0.5)
    axes[1].set_title('Attention (CLS \u2192 patches)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

for i in range(len(images)):
    show_attention(model, images[i], originals[i], i)

## Patch Feature PCA

In [ ]:
from sklearn.decomposition import PCA

feats = patch_features[0].detach().cpu().numpy()  # [N, 256]
n_patches = int(feats.shape[0] ** 0.5)

pca = PCA(n_components=3)
rgb = pca.fit_transform(feats)
rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-8)
rgb = rgb.reshape(n_patches, n_patches, 3)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(originals[0])
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(rgb)
axes[1].set_title('PCA of Patch Features (RGB)')
axes[1].axis('off')
plt.tight_layout()
plt.show()